# 16 — Test: Milestone 2 pipeline on 1,000 papers (Colab)

Runs the real `15_run_parsing.ipynb` on `TEST_PAPERS` = 1,000 papers of the group's data on Drive (spread over
the four ranges, plus the hard cases of `15`), with `WORKERS` = 6 supervised processes, then checks the result.

**The real data is not touched.** The test works in `/content/m2_test` (the Colab disk): `data/raw` there is a
link to `MyDrive/arxiv-project/data/raw`, which is only read; the discovery manifest and the download receipts
are copied. `data/parsed`, the tables, the logs and the resume index of the test stay in `/content/m2_test`, and
the results are copied to `MyDrive/arxiv-project/test_runs/m2_<N>_papers/` at the end.

What it checks:
1. `15` itself: its `Checks:` line (invariants, status log complete) and its statistics and benchmark;
2. the three Parquet tables against the statement: column names and order, types, no null key, unique and
   well-formed `element_id` (prefix of its kind, folder of its paper), every `parent_id` in `elements`, the
   document as the only root of each version, hierarchy rows for exactly the `ok` versions, unique BibTeX keys;
3. resume: a second run would parse nothing;
4. the time limits, on Linux (they cannot be tested on Windows): a paper stuck in Python code and one stuck in
   a regular expression (C code), both stopped by `SIGALRM` on Colab, and one whose worker is killed (as the
   system does out of memory), given up by the supervisor. Section 5 runs last, on 6 papers, after the results
   are saved.

**Before running:** the code is cloned from GitHub (branch `m2-pipeline`), so the last changes must be pushed.
**Time:** mostly Drive reads; the progress bar of `15` shows the ETA. If Colab disconnects but the runtime is
still alive, set `FRESH = False` and run all cells again: the test resumes. After a runtime reset, start again
with `FRESH = True`.

In [ ]:
# On Colab: clone or update the branch, install packages, work from src/ (the notebooks use relative paths).
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

BRANCH = "m2-pipeline"
if ON_COLAB:
    REPO = Path("/content/intro-ds-arxiv-project")
    URL = "https://github.com/thinhelzan842/intro-ds-arxiv-project.git"
    if not (REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin",
                        f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    os.chdir(REPO / "src")
    print("Code in use:", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"],
                                         capture_output=True, text=True).stdout.strip())

TEST_PAPERS = 1000                  # papers of the test (plus the hard cases of 15)
WORKERS = 6                         # supervised parsing processes
FRESH = True                        # True: start from an empty test folder; False: resume an interrupted test
TEST_ROOT = Path("/content/m2_test") if ON_COLAB else Path("../test_runs/m2_test")
RUN_TIME_LIMIT_TEST = True          # section 4: stuck and killed workers (Linux only)

get_ipython().run_line_magic("run", "-i ./00_config.ipynb")   # mounts Drive, PROJECT_ROOT
RESULTS = PROJECT_ROOT / "test_runs" / f"m2_{TEST_PAPERS}_papers"
print("Test folder:", TEST_ROOT, "| results will be copied to:", RESULTS)

## 1. The test folder

`data/raw` is a link to the real raw folders (read only). The manifest gives the paper list and the version
counts; the download receipts give the reason of each `empty_source` version. Nothing else is shared with the
real run: `15` writes `data/parsed`, `data/m2`, `logs` and `state/m2` under `TEST_ROOT`.

In [ ]:
raw_link = TEST_ROOT / "data" / "raw"
if FRESH and TEST_ROOT.exists():
    if raw_link.is_symlink():
        raw_link.unlink()   # remove the link itself first: the real data/raw is never entered
    shutil.rmtree(TEST_ROOT)
(TEST_ROOT / "data").mkdir(parents=True, exist_ok=True)
(TEST_ROOT / "state" / "download").mkdir(parents=True, exist_ok=True)
(TEST_ROOT / "logs").mkdir(parents=True, exist_ok=True)

if not raw_link.exists():
    raw_link.symlink_to(DATA_DIR, target_is_directory=True)   # DATA_DIR of 00_config: the real data/raw
for source, target in ((STATE_DIR / "discovery_manifest.json", TEST_ROOT / "state" / "discovery_manifest.json"),
                       (STATE_DIR / "download" / "receipts.jsonl", TEST_ROOT / "state" / "download" / "receipts.jsonl"),
                       (STATE_DIR / "download" / "errors.jsonl", TEST_ROOT / "state" / "download" / "errors.jsonl")):
    if source.is_file() and not target.exists():
        shutil.copyfile(source, target)
print("raw ->", os.readlink(raw_link), "|", sum(1 for _ in os.scandir(raw_link)), "folders")
print("copied:", [p.name for p in (TEST_ROOT / "state").rglob("*") if p.is_file()])

## 2. Run `15_run_parsing.ipynb`

`CLEAN_OLD_PARSED` stays `False`: with `data/raw` linked to the real folders it would delete files there.
`15` ends with an `AssertionError` if one of its checks fails; this cell prints it and the test goes on, so the
results are still checked and saved.

In [ ]:
M2_OVERRIDES = dict(M2_ROOT=str(TEST_ROOT), TEST_PAPERS=TEST_PAPERS, WORKERS=WORKERS, FRESH_RUN=FRESH,
                    CLEAN_OLD_PARSED=False, MERGE_TABLES=True, RETRY_FAILED=False, FLUSH_DRIVE_AT_END=False)
run_started = time.time()
RUN_ERROR = None
try:
    get_ipython().run_line_magic("run", "-i ./15_run_parsing.ipynb")
except AssertionError as err:   # the checks of 15 failed: keep going, the cells below show the details
    RUN_ERROR = f"{type(err).__name__}: {err}"
    print("15 reported:", RUN_ERROR)
print(f"\n15 finished in {(time.time() - run_started) / 60:.1f} min")

## 3. The three tables against the statement

Read back from `data/m2/*.parquet`. Columns must be exactly the statement's, in its order; `kind` and
`entry_type` are dictionary-encoded strings, `version` and `year` integers, `authors` a list of strings.

In [ ]:
import pandas as pd


def check_tables(elements, hierarchy, bibtex, statuses):
    """Broken rules of the three tables (empty list if none), and a few counts."""
    problems = []

    def bad(count, message):
        if count:
            problems.append(f"{int(count):,} {message}")

    for column in ("element_id", "arxiv_id", "kind"):
        bad(elements[column].isna().sum(), f"null {column} in elements")
    for column in ("arxiv_id", "version", "element_id", "parent_id"):
        bad(hierarchy[column].isna().sum(), f"null {column} in hierarchy")
    bad(elements["element_id"].duplicated().sum(), "duplicate element_id in elements")
    parts = elements["element_id"].str.extract(r"^(\d{4}-\d{4,5})#([a-z]+?)\d*$")
    bad(parts[0].isna().sum(), "element_id not of the form <yymm-id>#<prefix><n>")
    same_paper = parts[0].str.replace("-", ".", n=1, regex=False) == elements["arxiv_id"]
    bad((~same_paper).sum(), "element_id whose folder is not its arxiv_id")
    bad((parts[1] != elements["kind"].astype(str).map(KIND_PREFIX)).sum(), "element_id prefix not matching its kind")
    known = set(elements["element_id"])
    bad((~hierarchy["element_id"].isin(known)).sum(), "hierarchy element_id not in elements")
    bad((~hierarchy["parent_id"].isin(known)).sum(), "hierarchy parent_id not in elements")
    bad(hierarchy["element_id"].str.endswith("#doc").sum(), "document roots listed as a child")
    wrong_roots = 0
    for (arxiv_id, version), rows in hierarchy.groupby(["arxiv_id", "version"], sort=False):
        if set(rows["parent_id"]) - set(rows["element_id"]) != {arxiv_id.replace(".", "-", 1) + "#doc"}:
            wrong_roots += 1
    bad(wrong_roots, "(paper, version) whose only root is not the document")
    ok = statuses[statuses["status"] == "ok"]
    ok_keys = set(zip(ok["arxiv_id"], ok["version"].astype(int)))
    # A tree that is the document alone (a body without text, e.g. a LaTeX wrapper of a PDF) has no
    # hierarchy row, since the root has none: listed below, not a broken rule.
    alone = ok[ok["n_nodes"].astype(int) <= 1]
    alone_keys = set(zip(alone["arxiv_id"], alone["version"].astype(int)))
    have_keys = set(zip(hierarchy["arxiv_id"], hierarchy["version"].astype(int)))
    bad(len(ok_keys - have_keys - alone_keys), "ok versions with content but without hierarchy rows")
    bad(len(have_keys - ok_keys), "versions with hierarchy rows that are not ok")
    bad(bibtex.duplicated(["arxiv_id", "citation_key"]).sum(), "duplicate (arxiv_id, citation_key) in bibtex")
    bad((bibtex["citation_key"].fillna("") == "").sum(), "empty citation_key in bibtex")
    years = bibtex["year"].dropna()
    bad(((years < 1500) | (years > 2100)).sum(), "bibtex year outside 1500-2100")
    counts = {
        "papers": int(elements["arxiv_id"].nunique()),
        "versions_with_rows": len(have_keys),
        "document_only_versions": sorted(f"{a} v{v}" for a, v in alone_keys),
        "rows": {"elements": len(elements), "hierarchy": len(hierarchy), "bibtex": len(bibtex)},
        "elements_by_kind": {str(k): int(n) for k, n in elements["kind"].astype(str).value_counts().items()},
    }
    return problems, counts


import pyarrow as pa
import pyarrow.parquet as pq

STATEMENT = {"elements": list(ELEMENT_COLUMNS), "hierarchy": list(HIERARCHY_COLUMNS), "bibtex": list(BIBTEX_COLUMNS)}
KINDS = {"element_id": "string", "arxiv_id": "string", "content": "string", "kind": "dictionary", "version": "integer",
         "parent_id": "string", "citation_key": "string", "entry_type": "dictionary", "title": "string",
         "authors": "list of strings", "year": "integer", "doi": "string", "journal": "string", "url": "string"}


def check_schema(name, schema):
    problems = [] if schema.names == STATEMENT[name] else [f"{name}: columns {schema.names}, statement {STATEMENT[name]}"]
    for field in schema:
        kind, t = KINDS.get(field.name), field.type
        ok = {"string": pa.types.is_string(t), "integer": pa.types.is_integer(t),
              "dictionary": pa.types.is_dictionary(t) and pa.types.is_string(t.value_type),
              "list of strings": pa.types.is_list(t) and pa.types.is_string(t.value_type)}.get(kind, True)
        if not ok:
            problems.append(f"{name}.{field.name}: {t}, expected {kind}")
    return problems


tables, table_report = {}, []
for name in STATEMENT:
    path = M2_DATA / f"{name}.parquet"
    if not path.is_file():
        table_report.append(f"{path} is missing")
        continue
    table_report += check_schema(name, pq.read_schema(path))
    tables[name] = pq.read_table(path).to_pandas()
statuses = pd.read_csv(STATUS_LOG, dtype={"arxiv_id": str})
counts = {}
if len(tables) == 3:
    problems, counts = check_tables(tables["elements"], tables["hierarchy"], tables["bibtex"], statuses)
    table_report += problems

done_now = read_done_index()
still_todo = [f for f in PAPERS if f not in done_now and f not in read_failed_index()]
resume_ok = not still_todo
given_up = read_failed_index()
print("Table checks:", "all passed" if not table_report else f"{len(table_report)} problems")
for line in table_report[:20]:
    print("  ", line)
print("Counts:", json.dumps(counts.get("rows", {})), "| papers:", counts.get("papers"))
print("Versions whose tree is the document alone (no text, so no hierarchy row):",
      counts.get("document_only_versions"))
print("Resume: a second run would parse", len(still_todo), "papers", "(OK)" if resume_ok else still_todo[:10])
print("Timed out / lost worker:", len(given_up), list(given_up.items())[:5])
errors = statuses[statuses["status"] == "error"]
print("Versions in error:", len(errors))
if len(errors):
    print(errors["error"].str.slice(0, 90).value_counts().head(10).to_string())
slowest = sorted(((s["seconds"], f) for f, s in done_now.items() if s.get("seconds")), reverse=True)[:10]
print("Slowest papers (s):", [(f, round(t, 1)) for t, f in slowest])

## 4. Save the results to Drive

`MyDrive/arxiv-project/test_runs/m2_<N>_papers/`: `report.json` (the checks above), `logs/` (status log,
`m2_stats.json`, `m2_benchmark.json`, RAM samples), `state_m2/` (resume and failed indexes), the three tables and
20 files of `data/parsed/` as samples.

In [ ]:
if RESULTS.exists():
    shutil.rmtree(RESULTS)
RESULTS.mkdir(parents=True)
shutil.copytree(TEST_ROOT / "logs", RESULTS / "logs")
shutil.copytree(TEST_ROOT / "state" / "m2", RESULTS / "state_m2", ignore=shutil.ignore_patterns("*.lock"))
if M2_DATA.is_dir():
    shutil.copytree(M2_DATA, RESULTS / "tables")
(RESULTS / "parsed_samples").mkdir()
for path in sorted(PARSED_DIR.glob("*.json"))[:20]:
    shutil.copyfile(path, RESULTS / "parsed_samples" / path.name)
stats_doc = read_json(M2_STATS) or {}
bench_doc = read_json(M2_BENCHMARK) or {}
report = {
    "utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "code": subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip(),
    "parser_version": PARSER_VERSION, "implementation": IMPLEMENTATION,
    "test_papers": len(PAPERS), "workers": WORKERS, "run_error": RUN_ERROR,
    "table_problems": table_report, "counts": counts, "resume_ok": resume_ok,
    "timed_out_or_lost": given_up, "versions_in_error": int(len(errors)),
    "status": stats_doc.get("status"), "parse_success_rate_of_tex_versions_pct":
        stats_doc.get("parse_success_rate_of_tex_versions_pct"),
    "benchmark": {k: v for k, v in bench_doc.items() if k != "ram_timeseries"},
    "slowest": [(f, round(t, 2)) for t, f in slowest],
}
(RESULTS / "report.json").write_text(json.dumps(report, indent=1, ensure_ascii=False), encoding="utf-8")
print("Saved to", RESULTS)
print(json.dumps({k: report[k] for k in ("run_error", "table_problems", "resume_ok", "versions_in_error")}, indent=1))
print(json.dumps(report["benchmark"], indent=1))

## 5. Time limits, for real on Linux

Six papers of the test are parsed again with `PAPER_TIMEOUT = 5` s and `SUPERVISOR_GRACE = 10` s, three of them
made to fail on purpose:

| Paper | Injected | Expected |
|---|---|---|
| 1st | an endless Python loop | `timeout: still parsing after 5 s` (SIGALRM, the worker goes on) |
| 2nd | `re.match(r"(a+)+$", "a" * 64 + "b")`, hours in C code | `timeout: still parsing after 5 s` (SIGALRM reaches the regex engine on Colab, seen on 19 Nov); the supervisor's message after 15 s is also accepted |
| 3rd | the worker kills itself (SIGKILL, as out of memory) | `timeout: no answer from its worker after 15 s` (supervisor) |
| 4th–6th | nothing | parsed normally |

It runs after the results are saved, since it adds lines to the test's logs. About 20–30 s.

In [ ]:
import re
import signal as _signal

if RUN_TIME_LIMIT_TEST and ON_COLAB:
    ids = list(PAPERS)[:6]
    HANG_PYTHON, HANG_REGEX, KILLED = ids[:3]
    _real_parse_paper = parse_paper

    def parse_paper(pdir):   # replaces 13's for this test only; the workers fork after this definition
        name = Path(pdir).name
        if name == HANG_PYTHON:
            while True:
                pass
        if name == HANG_REGEX:
            re.match(r"(a+)+$", "a" * 64 + "b")
        if name == KILLED:
            os.kill(os.getpid(), _signal.SIGKILL)
        return _real_parse_paper(pdir)

    PAPER_TIMEOUT, SUPERVISOR_GRACE = 5, 10
    started = time.monotonic()
    try:
        _monitor, _session, _counts = run_parsing(ids)
        _monitor.stop()
        _session.finish({})
    finally:
        parse_paper = _real_parse_paper
        PAPER_TIMEOUT, SUPERVISOR_GRACE = 600, 60
    failed_now = read_failed_index()
    verdict = {
        "seconds": round(time.monotonic() - started, 1),
        "python_loop": failed_now.get(HANG_PYTHON, "NOT given up"),
        "regex_in_c": failed_now.get(HANG_REGEX, "NOT given up"),
        "killed_worker": failed_now.get(KILLED, "NOT given up"),
        "counts": dict(_counts),
    }
    passed = ("still parsing after 5 s" in verdict["python_loop"]
              and ("still parsing after 5 s" in verdict["regex_in_c"] or "no answer" in verdict["regex_in_c"])
              and "no answer" in verdict["killed_worker"] and _counts.get("timeout") == 3)
    verdict["passed"] = passed
    print(json.dumps(verdict, indent=1))
    (RESULTS / "time_limit_test.json").write_text(json.dumps(verdict, indent=1), encoding="utf-8")
    print("TIME LIMIT TEST:", "PASSED" if passed else "FAILED")
else:
    print("Skipped (RUN_TIME_LIMIT_TEST is False, or not on Colab).")

## 6. What to send back

`MyDrive/arxiv-project/test_runs/m2_<N>_papers/report.json` and `time_limit_test.json` (small), and if a check
failed, `logs/m2_parse_status.csv`. The test folder `/content/m2_test` disappears with the runtime; the real
`data/`, `logs/` and `state/` of the project were only read.